# Diagnostic ablation results

This notebook converts deterministic evaluation artifacts from completed experiment runs into report tables and figures. It reports retrieval, graph topology, structured argument generation, and symbolic resolution as separate paired effects.

In [ ]:
from pathlib import Path
import json
import math
import os

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "clinical_cds").is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / "output/.matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUTPUT_DIR = REPO_ROOT / "output/notebook_artifacts/ablation_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE_ORDER = [
    "direct",
    "flat_rag",
    "graph_rag",
    "structured_argument",
    "symbolic_argument",
]
MODE_LABELS = {
    "direct": "Direct",
    "flat_rag": "Flat RAG",
    "graph_rag": "Graph RAG",
    "structured_argument": "Structured argument",
    "symbolic_argument": "Symbolic argument",
}
MODE_COLORS = {
    "direct": "#687078",
    "flat_rag": "#3478A5",
    "graph_rag": "#D17A22",
    "structured_argument": "#2A7F62",
    "symbolic_argument": "#8B3F36",
}

def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else REPO_ROOT / path

def configured_runs():
    multiple = os.environ.get("EXPERIMENT_DIRS")
    single = os.environ.get("EXPERIMENT_DIR")
    if multiple:
        specs = []
        for item in multiple.split(os.pathsep):
            if not item.strip():
                continue
            if "=" in item:
                label, value = item.split("=", 1)
            else:
                value = item
                label = Path(value).name
            specs.append((label.strip(), resolve_path(value.strip()), True))
        return specs
    if single:
        return [(Path(single).name, resolve_path(single), True)]
    return [
        (
            "direct_test",
            resolve_path(os.environ.get("DIRECT_EXPERIMENT_DIR", "output/experiments/direct_test")),
            bool(os.environ.get("DIRECT_EXPERIMENT_DIR")),
        ),
        (
            "direct_test_strict",
            resolve_path(os.environ.get("STRICT_EXPERIMENT_DIR", "output/experiments/direct_test_strict")),
            bool(os.environ.get("STRICT_EXPERIMENT_DIR")),
        ),
        (
            "direct_test_umls",
            resolve_path(os.environ.get("UMLS_EXPERIMENT_DIR", "output/experiments/direct_test_umls")),
            bool(os.environ.get("UMLS_EXPERIMENT_DIR")),
        ),
        (
            "medqa_test",
            resolve_path(os.environ.get("MEDQA_EXPERIMENT_DIR", "output/experiments/medqa_test")),
            bool(os.environ.get("MEDQA_EXPERIMENT_DIR")),
        ),
    ]

run_specs = []
seen_paths = set()
for label, path, required in configured_runs():
    resolved = path.resolve()
    if resolved in seen_paths:
        continue
    if not (path / "evaluation/mode_summary.csv").is_file():
        if required:
            raise FileNotFoundError(f"No completed evaluation found at {path}")
        continue
    seen_paths.add(resolved)
    run_specs.append((label, path))

if not run_specs:
    raise FileNotFoundError(
        "No completed experiment runs were found. Run the CLI experiments in the README "
        "or set EXPERIMENT_DIR to a run containing evaluation/mode_summary.csv."
    )

run_specs

## Load completed runs

In [ ]:
def read_optional_csv(path):
    if not path.is_file() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

summary_frames = []
comparison_frames = []
case_frames = []
inventory_rows = []

for run_label, run_dir in run_specs:
    manifest_path = run_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}
    metadata = {
        "run": run_label,
        "run_id": manifest.get("run_id", run_dir.name),
        "model_id": manifest.get("model_id", "not_recorded"),
        "prompt_version": manifest.get("prompt_version", "not_recorded"),
        "normalizer_id": manifest.get("normalizer_id", "not_recorded"),
        "top_k": manifest.get("top_k"),
    }
    summary = pd.read_csv(run_dir / "evaluation/mode_summary.csv")
    if "selective_accuracy" not in summary:
        summary["selective_accuracy"] = summary["accuracy"] / summary["coverage"].replace(0, np.nan)
    summary = summary.assign(**metadata)
    summary_frames.append(summary)

    case_metrics = pd.read_csv(run_dir / "evaluation/case_metrics.csv").assign(**metadata)
    case_frames.append(case_metrics)

    comparisons = read_optional_csv(run_dir / "evaluation/paired_comparisons.csv")
    if not comparisons.empty:
        comparison_frames.append(comparisons.assign(**metadata))

    inventory_rows.append(
        {
            **metadata,
            "run_directory": str(run_dir),
            "case_count": manifest.get("case_count", case_metrics["case_id"].nunique()),
            "prediction_count": manifest.get("prediction_count", len(case_metrics)),
            "error_count": manifest.get("error_count", int(case_metrics["error"].sum())),
        }
    )

summaries = pd.concat(summary_frames, ignore_index=True)
case_metrics = pd.concat(case_frames, ignore_index=True)
comparisons = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
run_inventory = pd.DataFrame(inventory_rows)
run_inventory.to_csv(OUTPUT_DIR / "run_inventory.csv", index=False)
run_inventory

## Main outcome table

In [ ]:
main_columns = [
    "run",
    "normalizer_id",
    "subset",
    "dataset",
    "mode",
    "n",
    "accuracy",
    "hierarchy_score",
    "coverage",
    "selective_accuracy",
    "citation_validity",
    "observation_f1",
    "retrieval_gold_coverage",
    "argument_schema_validity",
    "argument_evidence_validity",
    "valid_evidence_reference_fraction",
    "verifier_review_coverage",
    "symbolic_trace_fidelity",
    "argument_resolution_change_rate",
    "mean_latency_seconds",
    "error_rate",
]
for column in main_columns:
    if column not in summaries:
        summaries[column] = np.nan

main_results = summaries[main_columns].copy()
main_results.to_csv(OUTPUT_DIR / "main_results.csv", index=False)
main_results.to_latex(
    OUTPUT_DIR / "main_results.tex",
    index=False,
    float_format="%.3f",
)
main_results

## Argument and explanation quality

These measures evaluate structural validity, grounding against supplied patient and guideline identifiers, verifier completeness, and reproducibility of the symbolic decision from the exported trace.

In [ ]:
quality_metrics = [
    ("argument_schema_validity", "Schema validity"),
    ("argument_evidence_validity", "Evidence validity"),
    ("verifier_review_coverage", "Verifier coverage"),
    ("symbolic_trace_fidelity", "Trace fidelity"),
]
argument_quality = summaries[
    (summaries["subset"] == "all")
    & summaries["mode"].isin(["structured_argument", "symbolic_argument"])
][
    [
        "run",
        "dataset",
        "mode",
        "n",
        *[metric for metric, _ in quality_metrics],
        "valid_evidence_reference_fraction",
        "argument_resolution_change_rate",
    ]
].copy()
argument_quality.to_csv(OUTPUT_DIR / "argument_quality.csv", index=False)
argument_quality.to_latex(
    OUTPUT_DIR / "argument_quality.tex",
    index=False,
    float_format="%.3f",
)

figure, axes = plt.subplots(1, len(quality_metrics), figsize=(15.0, 4.2))
for axis, (metric, title) in zip(axes, quality_metrics, strict=True):
    rows = argument_quality.dropna(subset=[metric]).reset_index(drop=True)
    labels = [f"{row.run}\n{row.mode}" for row in rows.itertuples()]
    axis.bar(
        np.arange(len(rows)),
        rows[metric],
        color=[MODE_COLORS[mode] for mode in rows["mode"]],
        edgecolor="#202428",
        linewidth=0.4,
    )
    axis.set_xticks(np.arange(len(rows)))
    axis.set_xticklabels(labels, rotation=25, ha="right")
    axis.set_ylim(0, 1)
    axis.set_title(title)
    axis.grid(axis="y", color="#D8D8D8", linewidth=0.6)
    axis.set_axisbelow(True)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "argument_quality.png", dpi=220)
plt.show()
argument_quality

## Comparisons with the direct baseline

In [ ]:
comparison_columns = [
    "run",
    "dataset",
    "baseline_mode",
    "comparison_mode",
    "n",
    "accuracy_difference",
    "bootstrap_ci_low",
    "bootstrap_ci_high",
    "mcnemar_baseline_only_correct",
    "mcnemar_comparison_only_correct",
    "mcnemar_exact_p",
]
if comparisons.empty:
    baseline_comparisons = pd.DataFrame(columns=comparison_columns)
else:
    for column in comparison_columns:
        if column not in comparisons:
            comparisons[column] = np.nan
    baseline_comparisons = comparisons[comparison_columns].copy()

baseline_comparisons.to_csv(OUTPUT_DIR / "direct_baseline_comparisons.csv", index=False)
baseline_comparisons.to_latex(
    OUTPUT_DIR / "direct_baseline_comparisons.tex",
    index=False,
    float_format="%.3f",
)
baseline_comparisons

## Incremental component effects

These paired comparisons isolate flat retrieval, graph topology, structured reasoner-verifier argument generation, and deterministic symbolic resolution.

In [ ]:
EFFECTS = [
    ("retrieval_context", "direct", "flat_rag"),
    ("graph_topology", "flat_rag", "graph_rag"),
    ("structured_argumentation", "graph_rag", "structured_argument"),
    ("symbolic_resolution", "structured_argument", "symbolic_argument"),
]

def exact_mcnemar(left, right):
    left_only = int(((left == 1.0) & (right == 0.0)).sum())
    right_only = int(((left == 0.0) & (right == 1.0)).sum())
    discordant = left_only + right_only
    if discordant == 0:
        return left_only, right_only, 1.0
    tail = min(left_only, right_only)
    probability = sum(math.comb(discordant, index) for index in range(tail + 1)) / (2 ** discordant)
    return left_only, right_only, min(1.0, 2.0 * probability)

effect_rows = []
comparison_index = 0
for (run, dataset_name), rows in case_metrics.groupby(["run", "dataset"], sort=False):
    paired = rows.pivot_table(
        index="case_id",
        columns="mode",
        values="exact_match",
        aggfunc="first",
    )
    for effect, baseline_mode, comparison_mode in EFFECTS:
        if baseline_mode not in paired or comparison_mode not in paired:
            continue
        values = paired[[baseline_mode, comparison_mode]].dropna()
        if values.empty:
            continue
        left = values[baseline_mode].to_numpy(dtype=float)
        right = values[comparison_mode].to_numpy(dtype=float)
        differences = right - left
        rng = np.random.default_rng(17 + comparison_index)
        bootstrap_means = rng.choice(
            differences,
            size=(2000, len(differences)),
            replace=True,
        ).mean(axis=1)
        left_only, right_only, p_value = exact_mcnemar(left, right)
        effect_rows.append(
            {
                "run": run,
                "dataset": dataset_name,
                "effect": effect,
                "baseline_mode": baseline_mode,
                "comparison_mode": comparison_mode,
                "n": len(values),
                "baseline_accuracy": left.mean(),
                "comparison_accuracy": right.mean(),
                "accuracy_difference": differences.mean(),
                "bootstrap_ci_low": np.quantile(bootstrap_means, 0.025),
                "bootstrap_ci_high": np.quantile(bootstrap_means, 0.975),
                "mcnemar_baseline_only_correct": left_only,
                "mcnemar_comparison_only_correct": right_only,
                "mcnemar_exact_p": p_value,
            }
        )
        comparison_index += 1

effect_columns = [
    "run",
    "dataset",
    "effect",
    "baseline_mode",
    "comparison_mode",
    "n",
    "baseline_accuracy",
    "comparison_accuracy",
    "accuracy_difference",
    "bootstrap_ci_low",
    "bootstrap_ci_high",
    "mcnemar_baseline_only_correct",
    "mcnemar_comparison_only_correct",
    "mcnemar_exact_p",
]
incremental_effects = pd.DataFrame(effect_rows, columns=effect_columns)
incremental_effects.to_csv(OUTPUT_DIR / "incremental_mode_effects.csv", index=False)
if not incremental_effects.empty:
    incremental_effects.to_latex(
        OUTPUT_DIR / "incremental_mode_effects.tex",
        index=False,
        float_format="%.3f",
    )
incremental_effects

## Outcome figures

In [ ]:
all_cases = summaries[summaries["subset"] == "all"].copy()
groups = list(all_cases.groupby(["run", "dataset"], sort=False))
positions = np.arange(len(groups), dtype=float)
width = 0.19

figure, axis = plt.subplots(figsize=(max(8.5, 2.0 * len(groups)), 5.2))
for mode_index, mode in enumerate(MODE_ORDER):
    values = []
    for _, rows in groups:
        match = rows[rows["mode"] == mode]
        values.append(float(match["accuracy"].iloc[0]) if not match.empty else np.nan)
    axis.bar(
        positions + (mode_index - (len(MODE_ORDER) - 1) / 2) * width,
        values,
        width=width,
        color=MODE_COLORS[mode],
        edgecolor="#202428",
        linewidth=0.4,
        label=MODE_LABELS[mode],
    )
axis.set_xticks(positions)
axis.set_xticklabels([f"{key[0]}\n{key[1]}" for key, _ in groups])
axis.set_ylim(0, 1)
axis.set_ylabel("Exact diagnosis accuracy")
axis.set_title("Diagnostic ablation results")
axis.grid(axis="y", color="#D8D8D8", linewidth=0.6)
axis.set_axisbelow(True)
axis.legend(frameon=False, ncol=2)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "accuracy_by_condition.png", dpi=220)
plt.show()

In [ ]:
figure, axis = plt.subplots(
    figsize=(9.0, max(3.2, 0.55 * max(len(incremental_effects), 1)))
)
if incremental_effects.empty:
    axis.text(0.5, 0.5, "No complete adjacent mode pairs were available.", ha="center", va="center")
    axis.set_axis_off()
else:
    plot_rows = incremental_effects.reset_index(drop=True)
    y_positions = np.arange(len(plot_rows))
    values = plot_rows["accuracy_difference"].to_numpy()
    lower = values - plot_rows["bootstrap_ci_low"].to_numpy()
    upper = plot_rows["bootstrap_ci_high"].to_numpy() - values
    effect_colors = {
        "retrieval_context": "#3478A5",
        "graph_topology": "#D17A22",
        "structured_argumentation": "#2A7F62",
        "symbolic_resolution": "#8B3F36",
    }
    for index, row in plot_rows.iterrows():
        axis.errorbar(
            row["accuracy_difference"],
            index,
            xerr=[
                [row["accuracy_difference"] - row["bootstrap_ci_low"]],
                [row["bootstrap_ci_high"] - row["accuracy_difference"]],
            ],
            fmt="o",
            color=effect_colors[row["effect"]],
            ecolor=effect_colors[row["effect"]],
            capsize=4,
        )
    axis.axvline(0.0, color="#A33A2B", linewidth=1.0)
    axis.set_yticks(y_positions)
    axis.set_yticklabels(
        [f"{row.run}, {row.dataset}: {row.effect.replace('_', ' ')}" for row in plot_rows.itertuples()]
    )
    axis.set_xlabel("Paired accuracy difference")
    axis.grid(axis="x", color="#D8D8D8", linewidth=0.6)
    axis.set_axisbelow(True)
axis.set_title("Incremental component effects with 95% bootstrap intervals")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "incremental_mode_effects.png", dpi=220)
plt.show()

## Case-level execution checks

In [ ]:
case_checks = (
    case_metrics.groupby(["run", "dataset", "mode"], as_index=False)
    .agg(
        cases=("case_id", "count"),
        accuracy=("exact_match", "mean"),
        coverage=("covered", "mean"),
        errors=("error", "sum"),
        cache_fraction=("cache_hit", "mean"),
    )
)
case_checks.to_csv(OUTPUT_DIR / "case_level_checks.csv", index=False)
case_checks

Confidence intervals and McNemar tests are paired by case. Statistical uncertainty should be reported alongside point estimates; small smoke runs are pipeline checks and are not study results.